In [1]:
import sys
import os
import akshare as ak
from datetime import date, datetime
# 获取当前文件所在目录（B 文件夹），再上移一级变成 project/
current_dir = os.path.dirname(os.path.abspath("__file__"))
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from DataFetch.stock_basic import *
import pandas as pd
from sqlalchemy import create_engine
from common.utils import get_engine, code_add_suffix
import time
from plotly.subplots import make_subplots
engine = get_engine()

In [2]:
query = f"""
SELECT * FROM daily_kline
WHERE trade_date >= '2025-01-01' AND trade_date <= '2025-05-29'
ORDER BY trade_date DESC
"""
df = pd.read_sql(query, engine)


In [ ]:
import akshare as ak

data_lst = []
for ts_code in df.ts_code:
    if ts_code.endswith("BJ"):
        continue
    try:
        data_tmp = ak.stock_comment_detail_scrd_focus_em(symbol=ts_code.split(".")[0])
        data_tmp.columns = ["trade_date", "follow_user_cnt"]
        data_tmp["ts_code"] = ts_code
        data_lst.append(data_tmp)
    except:
        print(ts_code)

In [16]:
stock_comment_detail_scrd_focus_em_df.columns = ["trade_date", "follow_user_cnt"]

In [17]:
stock_comment_detail_scrd_focus_em_df["ts_code"] = code_add_suffix("688653")

In [26]:
df_f = pd.merge(df, stock_comment_detail_scrd_focus_em_df, on = ["trade_date", "ts_code"])

In [27]:
df_f.head(100)

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,follow_user_cnt
0,688653.SH,2025-05-29,11.27,11.63,11.27,11.63,11.28,0.35,3.1028,58217.1,66912.5,60.8
1,688653.SH,2025-05-28,11.52,11.55,11.21,11.28,11.46,-0.18,-1.5707,33472.1,37926.0,60.8
2,688653.SH,2025-05-27,11.66,11.66,11.36,11.46,11.50,-0.04,-0.3478,32929.8,37627.9,61.2
3,688653.SH,2025-05-26,11.31,11.51,11.31,11.50,11.32,0.18,1.5901,29112.3,33256.8,62.0
4,688653.SH,2025-05-23,11.57,11.62,11.32,11.32,11.65,-0.33,-2.8326,47777.4,54775.6,61.6
5,688653.SH,2025-05-22,11.45,12.04,11.43,11.65,11.53,0.12,1.0408,63618.4,75036.2,60.4
6,688653.SH,2025-05-21,11.73,11.73,11.44,11.53,11.66,-0.13,-1.1149,33788.2,39014.6,60.4
7,688653.SH,2025-05-20,11.45,11.66,11.38,11.66,11.45,0.21,1.8341,39839.6,46021.5,60.4
8,688653.SH,2025-05-19,11.49,11.54,11.32,11.45,11.40,0.05,0.4386,35026.8,40034.0,61.2
9,688653.SH,2025-05-16,11.38,11.54,11.32,11.40,11.38,0.02,0.1757,32846.4,37614.4,60.0


In [28]:
lag_cols = []
for lag_col in ["vol", "amount", "follow_user_cnt"]:
    lag_cols.append(f"{lag_col}_lag1")
    df_f[f"{lag_col}_lag1"] = df_f[lag_col].shift(1)


In [29]:
df_f.head(2)

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,follow_user_cnt,vol_lag1,amount_lag1,follow_user_cnt_lag1
0,688653.SH,2025-05-29,11.27,11.63,11.27,11.63,11.28,0.35,3.1028,58217.1,66912.5,60.8,NaN,NaN,NaN
1,688653.SH,2025-05-28,11.52,11.55,11.21,11.28,11.46,-0.18,-1.5707,33472.1,37926.0,60.8,58217.1,66912.5,60.8


In [30]:
df_f.dropna(subset=['pct_chg'] + lag_cols)[['pct_chg'] + lag_cols].corr()


,pct_chg,vol_lag1,amount_lag1,follow_user_cnt_lag1
pct_chg,1.000000,0.439348,0.453442,0.266955
vol_lag1,0.439348,1.000000,0.998496,0.309157
amount_lag1,0.453442,0.998496,1.000000,0.332872
follow_user_cnt_lag1,0.266955,0.309157,0.332872,1.000000
